In [26]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp

In [27]:
import os

print(os.listdir("Offroad_Segmentation_Training_Dataset"))


['train', 'val']


In [28]:
from torch.utils.data import Dataset
import os
from PIL import Image

class DesertDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform
        
        # store only actual image files
        self.images = [f for f in os.listdir(image_dir) if not f.startswith(".")]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]

        img_path = os.path.join(self.image_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)

        return image, mask


In [29]:
train_dataset = DesertDataset(train_image_path, train_mask_path)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

print("Train size:", len(train_dataset))


Train size: 2859


In [30]:
img, mask = train_dataset[0]
print(img.size)
print(mask.size)


(960, 540)
(960, 540)


In [31]:
import torchvision.transforms as T

transform = T.Compose([
    T.ToTensor()
])


In [32]:
train_dataset = DesertDataset(train_image_path, train_mask_path, transform=transform)


In [33]:
import torch

def iou_score(preds, masks, threshold=0.5):
    preds = torch.sigmoid(preds)          # convert logits to probabilities
    preds = (preds > threshold).float()  # convert to 0/1

    masks = masks.float()

    intersection = (preds * masks).sum()
    union = preds.sum() + masks.sum() - intersection

    if union == 0:
        return torch.tensor(1.0)

    return intersection / union


 

In [34]:
import torch
import torch.nn as nn
import torchvision.models as models

def get_model():
    model = models.segmentation.fcn_resnet50(weights="DEFAULT")
    
    # Change final classifier to 1 channel (binary segmentation)
    model.classifier[4] = nn.Conv2d(512, 1, kernel_size=1)
    
    return model



In [40]:
model = get_model().to(device)


In [42]:
from torch.utils.data import Dataset
import os
from PIL import Image
import torchvision.transforms as T
import torch

class DesertDataset(Dataset):
    def __init__(self, image_dir, mask_dir):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        
        self.images = [f for f in os.listdir(image_dir) if not f.startswith(".")]

        self.image_transform = T.ToTensor()
        self.mask_transform = T.ToTensor()

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]

        img_path = os.path.join(self.image_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")  # grayscale mask

        image = self.image_transform(image)
        mask = self.mask_transform(mask)

        # Ensure mask is binary (0 or 1)
        mask = (mask > 0).float()

        return image, mask


In [44]:
val_dataset = DesertDataset(val_image_path, val_mask_path)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False
)

print("Validation dataset size:", len(val_dataset))


Validation dataset size: 317


In [ ]:
model.eval()
val_iou = 0

with torch.no_grad():
    for images, masks in val_loader:
        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)['out']
        val_iou += iou_score(outputs, masks).item()

print("Validation IoU:", val_iou / len(val_loader))
